In [1]:
# --- import ---
import sqlite3
import pandas as pd

In [2]:
DB = "../sql/TTC_delay.db"
MODES = ["bus", "streetcar", "subway"]

union_sql = " UNION ALL ".join(f"""
    SELECT '{m.capitalize()}' AS transit_mode, source_id, date, month, quarter,
           day, hour, time, line, station AS location, bound, vehicle,
           code, description AS delay_reason, min_delay, min_gap, is_delay
    FROM {m}_delays""" for m in MODES)

conn = sqlite3.connect(DB)
conn.execute("DROP VIEW IF EXISTS view_all_delays")
conn.execute(f"CREATE VIEW view_all_delays AS {union_sql}")
conn.commit()

conn.close()
print("view_all_delays created.")

view_all_delays created.


In [3]:
conn = sqlite3.connect(DB)
conn.execute("DROP VIEW IF EXISTS view_fact_delays")
conn.execute("""
CREATE VIEW view_fact_delays AS
SELECT *,
    transit_mode || ' - ' || line AS line_label,

    CASE strftime('%w', date) WHEN '0' THEN 7
         ELSE CAST(strftime('%w', date) AS INTEGER) END AS day_num,
    CASE WHEN strftime('%w', date) IN ('0', '6') THEN 'Weekend'
         ELSE 'Weekday' END AS day_type,

    CASE WHEN hour <= 5  THEN 'Overnight' WHEN hour <= 9  THEN 'AM Peak'
         WHEN hour <= 15 THEN 'Midday'    WHEN hour <= 19 THEN 'PM Peak'
         ELSE 'Evening' END AS time_period,
    CASE WHEN hour <= 5  THEN 1 WHEN hour <= 9  THEN 2
         WHEN hour <= 15 THEN 3 WHEN hour <= 19 THEN 4
         ELSE 5 END AS period_num,

    CASE WHEN min_delay = 0  THEN 'No delay' WHEN min_delay <= 5  THEN '1-5 min'
         WHEN min_delay <= 15 THEN '6-15 min' WHEN min_delay <= 30 THEN '16-30 min'
         ELSE '30+ min' END AS severity,
    CASE WHEN min_delay = 0  THEN 1 WHEN min_delay <= 5  THEN 2
         WHEN min_delay <= 15 THEN 3 WHEN min_delay <= 30 THEN 4
         ELSE 5 END AS severity_num
FROM view_all_delays
""")
conn.commit()

print(pd.read_sql("PRAGMA table_info(view_fact_delays)", conn)["name"].tolist())
conn.close()

['transit_mode', 'source_id', 'date', 'month', 'quarter', 'day', 'hour', 'time', 'line', 'location', 'bound', 'vehicle', 'code', 'delay_reason', 'min_delay', 'min_gap', 'is_delay', 'line_label', 'day_num', 'day_type', 'time_period', 'period_num', 'severity', 'severity_num']


In [4]:
conn = sqlite3.connect(DB)
print(pd.read_sql("SELECT type, name FROM sqlite_master ORDER BY type, name", conn))

     type                      name
0   table                bus_delays
1   table          streetcar_delays
2   table             subway_delays
3    view           view_all_delays
4    view          view_bus_by_hour
5    view         view_bus_by_month
6    view        view_bus_by_reason
7    view          view_fact_delays
8    view    view_streetcar_by_hour
9    view   view_streetcar_by_month
10   view  view_streetcar_by_reason
11   view       view_subway_by_hour
12   view      view_subway_by_month
13   view     view_subway_by_reason


In [5]:
conn = sqlite3.connect("../sql/TTC_delay.db")

# rows per mode, date range
print(pd.read_sql("""
    SELECT transit_mode, COUNT(*) AS rows, MIN(date) AS first_date, MAX(date) AS last_date
    FROM view_fact_delays GROUP BY transit_mode""", conn))

# is source_id unique? (both numbers should match)
print(pd.read_sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT source_id) AS unique_ids FROM view_fact_delays", conn))

conn.close()

  transit_mode    rows           first_date            last_date
0          Bus  102257  2025-01-01 00:00:00  2026-08-31 00:00:00
1    Streetcar   23703  2025-01-01 00:00:00  2026-08-31 00:00:00
2       Subway   45475  2025-01-01 00:00:00  2026-08-31 00:00:00
     rows  unique_ids
0  171435      171435
